In [ ]:
import os
import random
import torch
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
from torchvision import transforms, models
from PIL import Image
from pathlib import Path
import numpy as np
import torch.optim as optim
from tqdm.auto import tqdm

# Fix the seed so your Train/Val splits never shift between runs
random.seed(42)
torch.manual_seed(42)

VISION_DIR = "../../../../datasets/vision"
SDV5_DIR = "../../../../datasets/genimage/imagenet_ai_0424_sdv5/train/ai"
VQDM_DIR = "../../../../datasets/genimage/imagenet_ai_0419_vqdm/train/ai"

DYNAMIC_POISONED_DIR = "../../../../datasets/prnu_injected_dynamic" 
ADDITIVE_POISONED_DIR = "../../../../datasets/prnu_injected_additive" 
SCALED_POISONED_DIR = "../../../../datasets/prnu_injected_scaled" 
VQDM_VAL_DIR = "../../../../datasets/genimage/imagenet_ai_0419_vqdm/val/ai" 
VACCINE_DIR = "../../../../datasets/prnu_injected_train_vaccine"

WEIGHTS_PATH = "../weights/spectre_srm_best.pth"

In [22]:
class SpectreDataset(Dataset):
    def __init__(self, vision_dir, sdv5_dir, vqdm_dir, poisoned_dir, split='train', transform=None):
        super().__init__()
        self.split = split
        self.transform = transform
        
        vision_path = Path(vision_dir)
        sdv5_path = Path(sdv5_dir)
        vqdm_path = Path(vqdm_dir)
        
        # 1. Deterministic Camera Splitting
        all_cameras = sorted([d for d in vision_path.iterdir() if d.is_dir()])
        assert len(all_cameras) == 35, f"Expected 35 camera folders, found {len(all_cameras)}"
        
        if split == 'train':
            assigned_cameras = all_cameras[:28]
        elif split == 'val':
            assigned_cameras = all_cameras[28:]
        else:
            raise ValueError("Split must be 'train' or 'val'")
            
        # 2. Load REAL (Class 0)
        self.real_paths = []
        for cam in assigned_cameras:
            self.real_paths.extend(list(cam.glob("*.jpg")) + list(cam.glob("*.png")))
            
        num_real = len(self.real_paths)
        
        # 3. Load FAKE (Class 1) - The Vaccination Protocol
        # We only vaccinate the training set. Validation remains pure to prevent data leakage.
        if split == 'train':
            num_poisoned = int(num_real * 0.15)
        else:
            num_poisoned = 0
            
        num_standard_fake = num_real - num_poisoned
        target_fakes_per_model = num_standard_fake // 2
        
        all_sdv5 = sorted(list(sdv5_path.glob("*.png")) + list(sdv5_path.glob("*.jpg")))
        all_vqdm = sorted(list(vqdm_path.glob("*.png")) + list(vqdm_path.glob("*.jpg")))
        
        sampled_sdv5 = random.sample(all_sdv5, target_fakes_per_model)
        sampled_vqdm = random.sample(all_vqdm, target_fakes_per_model)
        
        self.fake_paths = sampled_sdv5 + sampled_vqdm
        
        # Only inject the poisoned images if we are in the train split
        if num_poisoned > 0:
            poisoned_path = Path(poisoned_dir)
            all_poisoned = sorted(list(poisoned_path.glob("*.png")))
            assert len(all_poisoned) >= num_poisoned, f"Need {num_poisoned} poisoned images, found {len(all_poisoned)}."
            sampled_poisoned = random.sample(all_poisoned, num_poisoned)
            self.fake_paths.extend(sampled_poisoned)
            
        # 4. Final Aggregation
        self.all_paths = self.real_paths + self.fake_paths
        self.labels = [0] * len(self.real_paths) + [1] * len(self.fake_paths)
        
        print(f"[{split.upper()}] Dataset built: {len(self.real_paths)} Real (from {len(assigned_cameras)} cameras) vs {len(self.fake_paths)} Fake.")

    def __len__(self):
        return len(self.all_paths)

    def __getitem__(self, idx):
        img_path = self.all_paths[idx]
        label = self.labels[idx]
        
        try:
            # Force RGB, SRM will handle the frequency extraction
            img = Image.open(img_path).convert('RGB')
            if self.transform:
                img = self.transform(img)
            return img, torch.tensor(label, dtype=torch.float32)
        except Exception as e:
            print(f" [Error] Corrupt image {img_path}: {e}")
            # If an image fails to load, randomly pull another one to keep batch sizes stable
            return self.__getitem__(random.randint(0, len(self) - 1))

# ── Strict Forensic Transforms ────────────────────────────────────────────────
# DO NOT ADD RESIZE. DO NOT ADD GAUSSIAN BLUR.
train_transform = transforms.Compose([
    transforms.RandomCrop(256), 
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.ToTensor(),
])

val_transform = transforms.Compose([
    transforms.FiveCrop(256),
    transforms.Lambda(lambda crops: torch.stack([transforms.ToTensor()(crop) for crop in crops]))
])

# ── Quick Sanity Check ────────────────────────────────────────────────────────
# Added POISONED_DIR as the 4th argument
train_set = SpectreDataset(VISION_DIR, SDV5_DIR, VQDM_DIR, VACCINE_DIR, split='train', transform=train_transform)
val_set = SpectreDataset(VISION_DIR, SDV5_DIR, VQDM_DIR, VACCINE_DIR, split='val', transform=val_transform)

[TRAIN] Dataset built: 1400 Real (from 28 cameras) vs 1400 Fake.
[VAL] Dataset built: 350 Real (from 7 cameras) vs 350 Fake.


In [23]:
class SRMFilter(nn.Module):
    """
    A frozen High-Pass filter layer that destroys image semantics
    and isolates the high-frequency noise residual (PRNU & AI Artifacts).
    """
    def __init__(self):
        super().__init__()
        # Standard Laplacian High-Pass Kernel
        # We divide by 12.0 to normalize the energy and prevent gradient explosion
        kernel = np.array([
            [-1, -1, -1],
            [-1,  8, -1],
            [-1, -1, -1]
        ], dtype=np.float32) / 12.0 
        
        # Expand to 3 channels (RGB) for depthwise convolution
        kernel = np.stack([kernel, kernel, kernel], axis=0)
        kernel = np.expand_dims(kernel, axis=1)
        
        # requires_grad=False is CRITICAL. The network is not allowed to change this filter.
        self.weight = nn.Parameter(torch.from_numpy(kernel), requires_grad=False)
        
        # groups=3 ensures each color channel gets filtered independently
        self.conv = nn.Conv2d(3, 3, kernel_size=3, stride=1, padding=1, bias=False, groups=3)
        self.conv.weight = self.weight

    def forward(self, x):
        return self.conv(x)

class SpectreSRM(nn.Module):
    def __init__(self):
        super().__init__()
        
        # Layer 0: The Semantic Eraser
        self.srm = SRMFilter()
        
        # Layer 1: The Spatial Backbone (ResNet-18)
        # CRITICAL: weights=None. We are training from scratch.
        # ImageNet weights are tuned for finding edges of dogs and cars. We want pure noise analysis.
        self.backbone = models.resnet18(weights=None)
        
        # Modify the final layer for Binary Classification (Real vs Fake)
        num_ftrs = self.backbone.fc.in_features
        self.backbone.fc = nn.Linear(num_ftrs, 1)

    def forward(self, x):
        # 1. Strip the image down to pure noise residuals
        noise_map = self.srm(x)
        
        # 2. Analyze the spatial noise patterns
        logits = self.backbone(noise_map)
        return logits

# Quick Sanity Check to ensure dimensions map correctly
dummy_tensor = torch.randn(8, 3, 256, 256) # Batch of 8, RGB, 256x256 crops
model = SpectreSRM()
out = model(dummy_tensor)
print(f"SPECTRE_SRM Output Shape: {out.shape} (Expected: [8, 1])")
print("Architecture loaded successfully.")

SPECTRE_SRM Output Shape: torch.Size([8, 1]) (Expected: [8, 1])
Architecture loaded successfully.


In [24]:
# ── 1. Hyperparameters & Setup ────────────────────────────────────────────────
# We use a very low learning rate because the noise floor is fragile.
# If you hit this with a standard 1e-3, the gradients will explode and the model will guess randomly.
BATCH_SIZE = 32
EPOCHS = 20
LEARNING_RATE = 1e-4 
WEIGHT_DECAY = 1e-5

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Executing on: {device}")

# Assuming train_set and val_set were instantiated in Block 1
train_loader = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True, num_workers=4, pin_memory=True)
val_loader = DataLoader(val_set, batch_size=BATCH_SIZE, shuffle=False, num_workers=4, pin_memory=True)

# Instantiate the model from Block 2
model = SpectreSRM().to(device)

# ── 2. The Math (Loss & Optimizer) ────────────────────────────────────────────
# We use BCEWithLogitsLoss. NEVER use a Sigmoid + standard BCELoss. 
# BCEWithLogitsLoss combines them in a single mathematical step for extreme numerical stability.
criterion = nn.BCEWithLogitsLoss()

# AdamW aggressively penalizes large weights, preventing the ResNet from memorizing specific cameras.
optimizer = optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)

# ── 3. The Execution Loop ─────────────────────────────────────────────────────
best_val_acc = 0.0
save_dir = "weights"
os.makedirs(save_dir, exist_ok=True)

print("Commencing SPECTRE_SRM Training...")

for epoch in range(EPOCHS):
    # --- TRAINING PHASE ---
    model.train()
    running_loss = 0.0
    correct_train = 0
    total_train = 0
    
    train_bar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS} [Train]")
    for images, labels in train_bar:
        images, labels = images.to(device), labels.to(device).unsqueeze(1) # Labels need to be [batch, 1]
        
        optimizer.zero_grad()
        
        logits = model(images)
        loss = criterion(logits, labels)
        
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item() * images.size(0)
        
        # Calculate Accuracy
        # Sigmoid converts logits to probabilities. > 0.5 is Class 1 (Fake)
        preds = (torch.sigmoid(logits) > 0.5).float()
        correct_train += (preds == labels).sum().item()
        total_train += labels.size(0)
        
        train_bar.set_postfix(loss=loss.item())

    train_epoch_loss = running_loss / total_train
    train_epoch_acc = correct_train / total_train

    # --- VALIDATION PHASE ---
    model.eval()
    val_loss = 0.0
    correct_val = 0
    total_val = 0
    
    # We do not track gradients here. It is a strict read-only test.
    with torch.no_grad():
        val_bar = tqdm(val_loader, desc=f"Epoch {epoch+1}/{EPOCHS} [Val]  ")
        for images, labels in val_bar:
            images, labels = images.to(device), labels.to(device).unsqueeze(1)
            
            # images shape is currently: [BatchSize, 5, 3, 256, 256]
            bs, n_crops, c, h, w = images.size()
            
            # 1. Collapse the crops into the batch dimension: [BatchSize * 5, 3, 256, 256]
            images_collapsed = images.view(-1, c, h, w)
            
            # 2. Run the model on all crops simultaneously
            logits_collapsed = model(images_collapsed)
            
            # 3. Un-collapse the logits back to [BatchSize, 5, 1]
            logits_uncollapsed = logits_collapsed.view(bs, n_crops, -1)
            
            # 4. Average the logits across the 5 crops (dim=1)
            # If even one crop is highly "Fake", it pulls the average logit up.
            avg_logits = logits_uncollapsed.mean(dim=1)
            
            loss = criterion(avg_logits, labels)
            
            val_loss += loss.item() * bs
            preds = (torch.sigmoid(avg_logits) > 0.5).float()
            correct_val += (preds == labels).sum().item()
            total_val += bs

    val_epoch_loss = val_loss / total_val
    val_epoch_acc = correct_val / total_val
    
    print(f"Epoch {epoch+1} Summary:")
    print(f"  Train -> Loss: {train_epoch_loss:.4f} | Acc: {train_epoch_acc:.4f}")
    print(f"  Val   -> Loss: {val_epoch_loss:.4f} | Acc: {val_epoch_acc:.4f}")
    
    # --- CHECKPOINTING ---
    if val_epoch_acc > best_val_acc:
        best_val_acc = val_epoch_acc
        save_path = os.path.join(save_dir, "spectre_srm_best.pth")
        torch.save(model.state_dict(), save_path)
        print(f"  [*] New best model saved to {save_path} with Acc: {best_val_acc:.4f}\n")
    else:
        print("\n")

print(f"Training Complete. Best Validation Accuracy: {best_val_acc:.4f}")

Executing on: cuda
Commencing SPECTRE_SRM Training...


Epoch 1/20 [Val]  : 100%|██████████| 22/22 [00:07<00:00,  2.85it/s]


Epoch 1 Summary:
  Train -> Loss: 0.1141 | Acc: 0.9554
  Val   -> Loss: 0.1038 | Acc: 0.9357
  [*] New best model saved to weights/spectre_srm_best.pth with Acc: 0.9357



Epoch 2/20 [Val]  : 100%|██████████| 22/22 [00:08<00:00,  2.65it/s]


Epoch 2 Summary:
  Train -> Loss: 0.0422 | Acc: 0.9854
  Val   -> Loss: 0.0072 | Acc: 0.9957
  [*] New best model saved to weights/spectre_srm_best.pth with Acc: 0.9957



...

Epoch 18 Summary:
  Train -> Loss: 0.0075 | Acc: 0.9982
  Val   -> Loss: 0.0240 | Acc: 0.9886




Epoch 19/20 [Val]  : 100%|██████████| 22/22 [00:08<00:00,  2.63it/s]


Epoch 19 Summary:
  Train -> Loss: 0.0075 | Acc: 0.9982
  Val   -> Loss: 0.0057 | Acc: 0.9986




Epoch 20/20 [Val]  : 100%|██████████| 22/22 [00:08<00:00,  2.65it/s]

Epoch 20 Summary:
  Train -> Loss: 0.0102 | Acc: 0.9971
  Val   -> Loss: 0.0098 | Acc: 0.9971


Training Complete. Best Validation Accuracy: 0.9986


### Evaluate on Pure Val images

In [25]:
# Ensure this points to the exact folder containing your Dynamic PSNR poisoned images
# AND ensure they went through the SDXL inpainting step if you implemented that pipeline
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ── 1. Load the Flawless Judge ────────────────────────────────────────────────
model = SpectreSRM().to(device)
model.load_state_dict(torch.load(WEIGHTS_PATH, map_location=device))
model.eval()

# We MUST use the exact same transform used in validation. No resizing.
eval_transform = transforms.Compose([
    transforms.FiveCrop(256),
    transforms.Lambda(lambda crops: torch.stack([transforms.ToTensor()(crop) for crop in crops]))
])

# ── 2. Execute the Attack ─────────────────────────────────────────────────────
poisoned_paths = list(Path(VQDM_VAL_DIR).glob("*.png"))
if not poisoned_paths:
    print(f"ERROR: No .png images found in {VQDM_VAL_DIR}")

print(f"Commencing Evasion Attack on {len(poisoned_paths)} VDQM Val images")

bypassed_count = 0
total_confidence = 0.0

with torch.no_grad():
    for img_path in tqdm(poisoned_paths, desc="Attacking SPECTRE_SRM"):
        try:
            img = Image.open(img_path).convert('RGB')
            tensor = eval_transform(img).unsqueeze(0).to(device) 
            
            # tensor shape is currently: [1, 5, 3, 256, 256]
            bs, n_crops, c, h, w = tensor.size()
            
            # 1. Collapse the crops into the batch dimension: [5, 3, 256, 256]
            tensor_collapsed = tensor.view(-1, c, h, w)
            
            # 2. Run the model on all 5 crops simultaneously
            logits_collapsed = model(tensor_collapsed)
            
            # 3. Un-collapse the logits back to [1, 5, 1]
            logits_uncollapsed = logits_collapsed.view(bs, n_crops, -1)
            
            # 4. Average the logits across the 5 crops
            # If even one crop detects the VAE grid, it pulls the average Fake probability up.
            avg_logits = logits_uncollapsed.mean(dim=1)
            
            prob = torch.sigmoid(avg_logits).item() # Probability of being FAKE
            
            total_confidence += prob
            
            # If the probability of being Fake drops below 50%, the model thinks it's Real.
            if prob < 0.5:
                bypassed_count += 1
                
        except Exception as e:
            print(f"Error processing {img_path.name}: {e}")

# ── 3. The Verdict ────────────────────────────────────────────────────────────
total_images = len(poisoned_paths)
evasion_rate = (bypassed_count / total_images) * 100 if total_images > 0 else 0
avg_fake_prob = total_confidence / total_images if total_images > 0 else 0

print("\n" + "="*50)
print("               ATTACK REPORT")
print("="*50)
print(f"Total Images Attacked : {total_images}")
print(f"Successful Bypasses   : {bypassed_count}")
print(f"Evasion Success Rate  : {evasion_rate:.2f}%")
print(f"Average 'Fake' Prob   : {avg_fake_prob:.4f} (Lower is better)")
print("="*50)

if evasion_rate > 50:
    print("\n[VERDICT]: CATASTROPHIC BYPASS. Your PRNU injection successfully blinded the forensic discriminator.")
else:
    print("\n[VERDICT]: ATTACK FAILED. The SRM filter saw through your payload. The AI artifacts are too loud.")

Commencing Evasion Attack on 6000 VDQM Val images


Attacking SPECTRE_SRM: 100%|██████████| 6000/6000 [00:58<00:00, 103.44it/s]


               ATTACK REPORT
Total Images Attacked : 6000
Successful Bypasses   : 52
Evasion Success Rate  : 0.87%
Average 'Fake' Prob   : 0.9898 (Lower is better)

[VERDICT]: ATTACK FAILED. The SRM filter saw through your payload. The AI artifacts are too loud.


### Evaluate on PRNU Injected Dynamic PSNR Val images

In [29]:
# Ensure this points to the exact folder containing your Dynamic PSNR poisoned images
# AND ensure they went through the SDXL inpainting step if you implemented that pipeline
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ── 1. Load the Flawless Judge ────────────────────────────────────────────────
model = SpectreSRM().to(device)
model.load_state_dict(torch.load(WEIGHTS_PATH, map_location=device))
model.eval()

# We MUST use the exact same transform used in validation. No resizing.
val_transform = transforms.Compose([
    transforms.FiveCrop(256),
    transforms.Lambda(lambda crops: torch.stack([transforms.ToTensor()(crop) for crop in crops]))
])

# ── 2. Execute the Attack ─────────────────────────────────────────────────────
poisoned_paths = list(Path(DYNAMIC_POISONED_DIR).glob("*.png"))
if not poisoned_paths:
    print(f"ERROR: No .png images found in {DYNAMIC_POISONED_DIR}")

print(f"Commencing Evasion Attack on {len(poisoned_paths)} PRNU Dynamic PSNR Injected images")

bypassed_count = 0
total_confidence = 0.0

with torch.no_grad():
    for img_path in tqdm(poisoned_paths, desc="Attacking SPECTRE_SRM"):
        try:
            img = Image.open(img_path).convert('RGB')
            tensor = val_transform(img).unsqueeze(0).to(device) 
            
            # tensor shape is currently: [1, 5, 3, 256, 256]
            bs, n_crops, c, h, w = tensor.size()
            
            # 1. Collapse the crops into the batch dimension: [5, 3, 256, 256]
            tensor_collapsed = tensor.view(-1, c, h, w)
            
            # 2. Run the model on all 5 crops simultaneously
            logits_collapsed = model(tensor_collapsed)
            
            # 3. Un-collapse the logits back to [1, 5, 1]
            logits_uncollapsed = logits_collapsed.view(bs, n_crops, -1)
            
            # 4. Average the logits across the 5 crops
            # If even one crop detects the VAE grid, it pulls the average Fake probability up.
            avg_logits = logits_uncollapsed.mean(dim=1)
            
            prob = torch.sigmoid(avg_logits).item() # Probability of being FAKE
            
            total_confidence += prob
            
            # If the probability of being Fake drops below 50%, the model thinks it's Real.
            if prob < 0.5:
                bypassed_count += 1
                
        except Exception as e:
            print(f"Error processing {img_path.name}: {e}")

# ── 3. The Verdict ────────────────────────────────────────────────────────────
total_images = len(poisoned_paths)
evasion_rate = (bypassed_count / total_images) * 100 if total_images > 0 else 0
avg_fake_prob = total_confidence / total_images if total_images > 0 else 0

print("\n" + "="*50)
print("               ATTACK REPORT")
print("="*50)
print(f"Total Images Attacked : {total_images}")
print(f"Successful Bypasses   : {bypassed_count}")
print(f"Evasion Success Rate  : {evasion_rate:.2f}%")
print(f"Average 'Fake' Prob   : {avg_fake_prob:.4f} (Lower is better)")
print("="*50)

if evasion_rate > 50:
    print("\n[VERDICT]: CATASTROPHIC BYPASS. Your PRNU injection successfully blinded the forensic discriminator.")
else:
    print("\n[VERDICT]: ATTACK FAILED. The SRM filter saw through your payload. The AI artifacts are too loud.")

Commencing Evasion Attack on 6000 PRNU Dynamic PSNR Injected images


Attacking SPECTRE_SRM: 100%|██████████| 6000/6000 [00:50<00:00, 117.69it/s]


               ATTACK REPORT
Total Images Attacked : 6000
Successful Bypasses   : 94
Evasion Success Rate  : 1.57%
Average 'Fake' Prob   : 0.9837 (Lower is better)

[VERDICT]: ATTACK FAILED. The SRM filter saw through your payload. The AI artifacts are too loud.


### Evaluate on PRNU Injected Additive Val images

In [27]:
# Ensure this points to the exact folder containing your Dynamic PSNR poisoned images
# AND ensure they went through the SDXL inpainting step if you implemented that pipeline
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ── 1. Load the Flawless Judge ────────────────────────────────────────────────
model = SpectreSRM().to(device)
model.load_state_dict(torch.load(WEIGHTS_PATH, map_location=device))
model.eval()

# We MUST use the exact same transform used in validation. No resizing.
val_transform = transforms.Compose([
    transforms.FiveCrop(256),
    transforms.Lambda(lambda crops: torch.stack([transforms.ToTensor()(crop) for crop in crops]))
])

# ── 2. Execute the Attack ─────────────────────────────────────────────────────
poisoned_paths = list(Path(ADDITIVE_POISONED_DIR).glob("*.png"))
if not poisoned_paths:
    print(f"ERROR: No .png images found in {ADDITIVE_POISONED_DIR}")

print(f"Commencing Evasion Attack on {len(poisoned_paths)} PRNU Additive Injected images")

bypassed_count = 0
total_confidence = 0.0

with torch.no_grad():
    for img_path in tqdm(poisoned_paths, desc="Attacking SPECTRE_SRM"):
        try:
            img = Image.open(img_path).convert('RGB')
            tensor = val_transform(img).unsqueeze(0).to(device) 
            
            # tensor shape is currently: [1, 5, 3, 256, 256]
            bs, n_crops, c, h, w = tensor.size()
            
            # 1. Collapse the crops into the batch dimension: [5, 3, 256, 256]
            tensor_collapsed = tensor.view(-1, c, h, w)
            
            # 2. Run the model on all 5 crops simultaneously
            logits_collapsed = model(tensor_collapsed)
            
            # 3. Un-collapse the logits back to [1, 5, 1]
            logits_uncollapsed = logits_collapsed.view(bs, n_crops, -1)
            
            # 4. Average the logits across the 5 crops
            # If even one crop detects the VAE grid, it pulls the average Fake probability up.
            avg_logits = logits_uncollapsed.mean(dim=1)
            
            prob = torch.sigmoid(avg_logits).item() # Probability of being FAKE
            
            total_confidence += prob
            
            # If the probability of being Fake drops below 50%, the model thinks it's Real.
            if prob < 0.5:
                bypassed_count += 1
                
        except Exception as e:
            print(f"Error processing {img_path.name}: {e}")

# ── 3. The Verdict ────────────────────────────────────────────────────────────
total_images = len(poisoned_paths)
evasion_rate = (bypassed_count / total_images) * 100 if total_images > 0 else 0
avg_fake_prob = total_confidence / total_images if total_images > 0 else 0

print("\n" + "="*50)
print("               ATTACK REPORT")
print("="*50)
print(f"Total Images Attacked : {total_images}")
print(f"Successful Bypasses   : {bypassed_count}")
print(f"Evasion Success Rate  : {evasion_rate:.2f}%")
print(f"Average 'Fake' Prob   : {avg_fake_prob:.4f} (Lower is better)")
print("="*50)

if evasion_rate > 50:
    print("\n[VERDICT]: CATASTROPHIC BYPASS. Your PRNU injection successfully blinded the forensic discriminator.")
else:
    print("\n[VERDICT]: ATTACK FAILED. The SRM filter saw through your payload. The AI artifacts are too loud.")

Commencing Evasion Attack on 6000 PRNU Additive Injected images


Attacking SPECTRE_SRM: 100%|██████████| 6000/6000 [00:49<00:00, 122.01it/s]


               ATTACK REPORT
Total Images Attacked : 6000
Successful Bypasses   : 61
Evasion Success Rate  : 1.02%
Average 'Fake' Prob   : 0.9879 (Lower is better)

[VERDICT]: ATTACK FAILED. The SRM filter saw through your payload. The AI artifacts are too loud.


### Evaluate on PRNU Injected Multiplicative Val images

In [28]:
# Ensure this points to the exact folder containing your Dynamic PSNR poisoned images
# AND ensure they went through the SDXL inpainting step if you implemented that pipeline
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ── 1. Load the Flawless Judge ────────────────────────────────────────────────
model = SpectreSRM().to(device)
model.load_state_dict(torch.load(WEIGHTS_PATH, map_location=device))
model.eval()

# We MUST use the exact same transform used in validation. No resizing.
val_transform = transforms.Compose([
    transforms.FiveCrop(256),
    transforms.Lambda(lambda crops: torch.stack([transforms.ToTensor()(crop) for crop in crops]))
])

# ── 2. Execute the Attack ─────────────────────────────────────────────────────
poisoned_paths = list(Path(SCALED_POISONED_DIR).glob("*.png"))
if not poisoned_paths:
    print(f"ERROR: No .png images found in {SCALED_POISONED_DIR}")

print(f"Commencing Evasion Attack on {len(poisoned_paths)} PRNU Multiplicative Injected images")

bypassed_count = 0
total_confidence = 0.0

with torch.no_grad():
    for img_path in tqdm(poisoned_paths, desc="Attacking SPECTRE_SRM"):
        try:
            img = Image.open(img_path).convert('RGB')
            tensor = val_transform(img).unsqueeze(0).to(device) 
            
            # tensor shape is currently: [1, 5, 3, 256, 256]
            bs, n_crops, c, h, w = tensor.size()
            
            # 1. Collapse the crops into the batch dimension: [5, 3, 256, 256]
            tensor_collapsed = tensor.view(-1, c, h, w)
            
            # 2. Run the model on all 5 crops simultaneously
            logits_collapsed = model(tensor_collapsed)
            
            # 3. Un-collapse the logits back to [1, 5, 1]
            logits_uncollapsed = logits_collapsed.view(bs, n_crops, -1)
            
            # 4. Average the logits across the 5 crops
            # If even one crop detects the VAE grid, it pulls the average Fake probability up.
            avg_logits = logits_uncollapsed.mean(dim=1)
            
            prob = torch.sigmoid(avg_logits).item() # Probability of being FAKE
            
            total_confidence += prob
            
            # If the probability of being Fake drops below 50%, the model thinks it's Real.
            if prob < 0.5:
                bypassed_count += 1
                
        except Exception as e:
            print(f"Error processing {img_path.name}: {e}")

# ── 3. The Verdict ────────────────────────────────────────────────────────────
total_images = len(poisoned_paths)
evasion_rate = (bypassed_count / total_images) * 100 if total_images > 0 else 0
avg_fake_prob = total_confidence / total_images if total_images > 0 else 0

print("\n" + "="*50)
print("               ATTACK REPORT")
print("="*50)
print(f"Total Images Attacked : {total_images}")
print(f"Successful Bypasses   : {bypassed_count}")
print(f"Evasion Success Rate  : {evasion_rate:.2f}%")
print(f"Average 'Fake' Prob   : {avg_fake_prob:.4f} (Lower is better)")
print("="*50)

if evasion_rate > 50:
    print("\n[VERDICT]: CATASTROPHIC BYPASS. Your PRNU injection successfully blinded the forensic discriminator.")
else:
    print("\n[VERDICT]: ATTACK FAILED. The SRM filter saw through your payload. The AI artifacts are too loud.")

Commencing Evasion Attack on 6000 PRNU Multiplicative Injected images


Attacking SPECTRE_SRM: 100%|██████████| 6000/6000 [00:47<00:00, 126.18it/s]


               ATTACK REPORT
Total Images Attacked : 6000
Successful Bypasses   : 186
Evasion Success Rate  : 3.10%
Average 'Fake' Prob   : 0.9671 (Lower is better)

[VERDICT]: ATTACK FAILED. The SRM filter saw through your payload. The AI artifacts are too loud.
